In [ ]:
import torch
from torch import nn
from d2l import torch as d2l
import pandas as pd
import os
from matplotlib import pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
d2l.DATA_HUB['ml-100k'] = (
    'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
    'cd4dcac4241c8a4ad7badc7ca635da8a69dddb83')

#@save
def read_data_ml100k():
    data_dir = d2l.download_extract('ml-100k')
    names = ['user_id', 'item_id', 'rating', 'timestamp']
    data = pd.read_csv(os.path.join(data_dir, 'u.data'), sep='\t',
                       names=names, engine='python')
    num_users = data.user_id.unique().shape[0]
    num_items = data.item_id.unique().shape[0]
    return data, num_users, num_items

In [ ]:
data, num_users, num_items = read_data_ml100k()
sparsity = 1 - len(data) / (num_users * num_items)
print(f'number of users: {num_users}, number of items: {num_items}')
print(f'matrix sparsity: {sparsity:f}')
print(data.head(5))


In [ ]:
fig, ax = plt.subplots(1, 1)
ax.hist(data['rating'], bins=5)
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.set_title('Distribution of Ratings in MovieLens 100K')
plt.show()

In [ ]:
def split_data_ml100k(data, num_users, num_items, split_mode='random', test_ratios=0.1):
    if (split_mode == 'seq_aware'):
        train_items, test_items, train_list = {}, {}, []
        for line in data.itertuples():
            u, i, rating, time = line[0], line[1], line[2], line[3]
            train_items.setdefault(u, []).append((u, i, rating, time))
            if u not in test_items or test_items[u][-1] < time:
                test_items[u] = (i, rating, time)

        for u in range(1, num_users + 1):
            train_list.extend(sorted(train_items[u], key=lambda k: k[3]))
        test_data = [(key, *values) for key, values in test_items.items()]
        train_data = [item for item in train_items.items() if item not in test_data]
        train_data = pd.DataFrame(train_data)
        test_data = pd.DataFrame(test_data)
    else:
        mask = [True if x == 1 else False for x in np.random.uniform(0, 1, (len(data))) < 1 - test_ratios]
        neg_mask = [not i for i in mask]
        train_data, test_data = data[mask], data[neg_mask]

    return train_data, test_data

In [ ]:
def load_data_ml100k(data, num_users, num_items, feedback='explicit'):
    users, items, scores = [], [], []
    inter = np.zeros((num_items, num_users)) if feedback == 'explicit' else {}
    for line in data.itertuples():
        user_index, item_index = int(line[1] - 1), int(line[2] - 1)
        score = int(line[3]) if feedback == 'explicit' else 1
        users.append(user_index)
        items.append(item_index)
        scores.append(score)

    if feedback == 'explicit':
        inter.setdefault(user_index, []).append(item_index)
    else:
        inter[item_index][user_index] = score

    return users, items, scores, inter

In [ ]:
def split_and_load_ml100k(split_mode='seq-aware', feedback='explicit',
                          test_ratio=0.1, batch_size=256):
    data, num_users, num_items = read_data_ml100k()
    train_data, test_data = split_data_ml100k(
        data, num_users, num_items, split_mode, test_ratio)
    train_u, train_i, train_r, _ = load_data_ml100k(
        train_data, num_users, num_items, feedback)
    test_u, test_i, test_r, _ = load_data_ml100k(
        test_data, num_users, num_items, feedback)

    train_u = torch.tensor(train_u, dtype=torch.long)
    train_i = torch.tensor(train_i, dtype=torch.long)
    train_r = torch.tensor(train_u, dtype=torch.long)

    test_u = torch.tensor(test_u, dtype=torch.long)
    test_i = torch.tensor(test_i, dtype=torch.long)
    test_r = torch.tensor(test_r, dtype=torch.long)

    train_set = TensorDataset(
       train_u, train_i, train_r 
    )
    
    test_set = TensorDataset(
       test_u, test_i, test_r 
    )

    train_iter = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=False)
    test_iter = DataLoader(test_set, batch_size=batch_size, shuffle=False)
    
    return num_users, num_items, train_iter, test_iter